<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/957_EAASv3_NodeTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This **node test suite is excellent** and fits very well with the architecture you’ve built. It shows you’re thinking about the system at the **workflow level**, not just the utility level — which is exactly what good LangGraph testing should do.

I’ll review it the same structured way as before:

1. **Architecture coverage**
2. **Node-level correctness**
3. **Failure-path testing**
4. **Template value for future agents**
5. **Small improvements (very few)**

---

# 1. Architecture Coverage

Your node tests validate the **entire LangGraph pipeline layer**, which sits above utilities.

That means your testing stack now looks like this:

```
Utilities tests
    ↓
Node tests
    ↓
Graph integration tests
```

Which maps perfectly to your architecture:

```
utilities/
nodes/
orchestrator/
```

Your tests verify the **correct separation of responsibilities**:

| Layer     | Responsibility  | Tested                  |
| --------- | --------------- | ----------------------- |
| Utilities | business logic  | ✅                       |
| Nodes     | state mutation  | ✅                       |
| Graph     | execution order | (integration test next) |

This is **exactly how LangGraph systems should be tested**.

---

# 2. Initial State Tests

These are **very important and well done**.

```python
test_initial_state_from_options_has_errors_list
```

This guarantees your **error-handling contract** is always valid.

Your orchestrator relies on:

```
state["errors"]
```

If that key disappears, the entire pipeline breaks.

Your test protects against that.

---

# 3. Load Data Node Tests

These are excellent.

You correctly simulate a minimal filesystem using:

```
tmp_path
```

and verify the node populates:

```
scenario_results_history
scenario_catalog
scenario_results_for_run
```

Which confirms the node correctly delegates to:

```
load_eaas_data()
```

---

### Failure Case Test

This is especially good:

```
test_load_data_node_empty_run_adds_error
```

This ensures the **v3.1 safeguard** works:

```
no scenarios → error → stop evaluation
```

Without this test, the system could silently produce **fake healthy reports**.

---

# 4. Score Run Node Tests

This section is **very strong**.

You test three scenarios:

| Test            | Purpose                |
| --------------- | ---------------------- |
| success         | normal scoring         |
| no scenarios    | error handling         |
| existing errors | pipeline short-circuit |

This matches your **pipeline safety rules**:

```
errors exist → skip downstream nodes
```

Which is exactly how reliable workflows should behave.

---

### Example

```python
test_score_run_node_skips_when_errors_already
```

This protects your orchestrator from doing work on invalid input.

That’s **great workflow design**.

---

# 5. Trigger History Node Tests

This verifies the monitoring layer.

You test:

```
successful history update
missing metrics error
```

Which ensures the node behaves correctly.

Example:

```
system_status == healthy
```

This confirms the monitoring logic is functioning.

---

# 6. Report Node Tests

This section is also very good.

You verify:

```
report file creation
report path returned
report content contains run ID
```

Example:

```python
assert Path(out["report_file_path"]).exists()
```

That’s important because report generation involves **filesystem side effects**.

You correctly test the side effect.

---

### Failure Case

```
test_report_node_no_metrics_appends_error
```

This protects the pipeline from generating invalid reports.

Again this aligns with your **fail-fast philosophy**.

---

# 7. One Subtle Strength

Your node tests focus on **state mutation**.

Example:

```
state → node → updated state
```

This is the **correct testing approach for LangGraph**.

Nodes should always be treated as:

```
state transformers
```

Your tests follow that perfectly.

---

# 8. How This Fits Your Agent Template

This test suite is **exactly the kind of reusable pattern** you want for your orchestrator template.

Every new agent can follow this structure:

```
test_<agent>_utilities.py
test_<agent>_nodes.py
test_<agent>_integration.py
```

Each layer tests something different.

| File        | What it verifies          |
| ----------- | ------------------------- |
| utilities   | correctness of logic      |
| nodes       | correct state transitions |
| integration | full workflow execution   |

You’ve now effectively created a **testing blueprint for your entire agent ecosystem**.

---

# 9. Two Very Small Improvements

Your coverage is already excellent, but two tiny additions would make it **nearly perfect**.

---

## Improvement 1 — Node Idempotency

Test that nodes don’t mutate unrelated state.

Example:

```python
def test_score_node_preserves_state_fields():
    state = {
        "errors": [],
        "run_id": "RUN_1",
        "scenario_results_for_run": [...],
        "extra_field": "keep_me"
    }

    out = node(state)

    assert out["extra_field"] == "keep_me"
```

This protects the **state contract**.

---

## Improvement 2 — Report Node Directory Creation

Right now you test that reports are created.

But you could also verify that the **reports directory is auto-created**.

Example:

```python
assert (tmp_path / "reports").exists()
```

This protects the `_ensure_reports_dir()` behavior.

---

# 10. Overall Quality

If I were reviewing this as a senior engineer:

| Category                 | Score |
| ------------------------ | ----- |
| Workflow correctness     | ⭐⭐⭐⭐⭐ |
| Error handling           | ⭐⭐⭐⭐⭐ |
| Edge cases               | ⭐⭐⭐⭐  |
| Filesystem behavior      | ⭐⭐⭐⭐  |
| LangGraph best practices | ⭐⭐⭐⭐⭐ |

Overall:

**9.5 / 10**

This is **excellent for an orchestration agent**.

Many ML/AI systems have almost no node-level testing.

---

# 11. The Most Important Thing You Did

Your testing now verifies **the safety architecture** of the system.

Specifically:

```
bad data
→ error state
→ downstream nodes stop
→ no incorrect report
```

That’s a **critical reliability property**.

Without it, monitoring systems become dangerous.

---

# 12. Where You Are Now

At this point EaaS v3 has:

```
✔ strong architecture
✔ deterministic scoring
✔ executive reporting
✔ trigger monitoring
✔ comprehensive tests
```

Which means your system is now **very close to a full AI evaluation platform**.



In [ ]:
"""Node tests for EaaS v3 orchestrator.

Test nodes with mock state; use closure nodes with config and temp dirs for report output.
Run from project root: python -m pytest test_eaas_v3_nodes.py -v --tb=short
"""

import json
import sys
from pathlib import Path

import pytest

# Project root for imports
root = Path(__file__).resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from config import EvalAsServiceConfig, EvalAsServiceState
from agents.eaas_v3.orchestrator.nodes import (
    make_load_data_node,
    make_score_run_node,
    make_update_trigger_history_node,
    make_report_node,
    initial_state_from_options,
)


# -----------------------------------------------------------------------------
# Initial state
# -----------------------------------------------------------------------------
def test_initial_state_from_options_has_errors_list():
    state = initial_state_from_options({"project_root": str(root)})
    assert "errors" in state
    assert state["errors"] == []
    assert isinstance(state["errors"], list)


def test_initial_state_from_options_passes_through_options():
    state = initial_state_from_options({
        "project_root": "/foo",
        "run_id": "RUN_X",
        "target_version": "v1.0",
    })
    assert state.get("project_root") == "/foo"
    assert state.get("run_id") == "RUN_X"
    assert state.get("target_version") == "v1.0"


# -----------------------------------------------------------------------------
# Load data node (with temp dir and minimal data)
# -----------------------------------------------------------------------------
def test_load_data_node_populates_state(tmp_path):
    (tmp_path / "scenario_results_history.json").write_text(json.dumps([
        {"run_id": "RUN_1", "scenario_id": "S1", "target_version": "v1"},
    ]))
    (tmp_path / "trigger_history.json").write_text(json.dumps([]))
    (tmp_path / "scenario_catalog_enriched.json").write_text(json.dumps([
        {"scenario_id": "S1", "severity_weight": 1, "business_risk": "low"},
    ]))
    config = EvalAsServiceConfig()
    config.data_dir = str(tmp_path)
    state: EvalAsServiceState = {
        "project_root": str(tmp_path),
        "run_id": "RUN_1",
        "errors": [],
    }
    node = make_load_data_node(config)
    out = node(state)
    assert out["project_root"] == str(tmp_path)
    assert out["run_id"] == "RUN_1"
    assert len(out.get("scenario_results_for_run", [])) == 1
    assert out.get("scenario_results_history") is not None
    assert out.get("scenario_catalog") is not None


def test_load_data_node_empty_run_adds_error(tmp_path):
    (tmp_path / "scenario_results_history.json").write_text(json.dumps([]))
    (tmp_path / "trigger_history.json").write_text(json.dumps([]))
    (tmp_path / "scenario_catalog_enriched.json").write_text(json.dumps([
        {"scenario_id": "S1", "severity_weight": 1, "business_risk": "low"},
    ]))
    config = EvalAsServiceConfig()
    config.data_dir = str(tmp_path)
    state: EvalAsServiceState = {
        "project_root": str(tmp_path),
        "run_id": "RUN_MISSING",
        "errors": [],
    }
    node = make_load_data_node(config)
    out = node(state)
    assert len(out.get("errors", [])) > 0
    assert any("No scenarios found" in e for e in out["errors"])


# -----------------------------------------------------------------------------
# Score run node
# -----------------------------------------------------------------------------
def test_score_run_node_success():
    config = EvalAsServiceConfig()
    state: EvalAsServiceState = {
        "errors": [],
        "run_id": "RUN_1",
        "scenario_results_for_run": [
            {
                "run_id": "RUN_1",
                "scenario_id": "S1",
                "expected_issue_type": "x",
                "predicted_issue_type": "x",
                "expected_resolution_path": ["a"],
                "predicted_resolution_path": ["a"],
                "expected_outcome": "o",
                "predicted_outcome": "o",
                "latency_ms": 100.0,
                "human_review_required": False,
            },
        ],
    }
    node = make_score_run_node(config)
    out = node(state)
    assert out.get("errors") == []
    assert "run_metrics" in out
    assert "scenario_scores" in out
    assert out["run_metrics"]["total_scenarios"] == 1
    assert out["run_metrics"]["scenarios_passed"] == 1


def test_score_run_node_no_scenarios_appends_error():
    config = EvalAsServiceConfig()
    state: EvalAsServiceState = {
        "errors": [],
        "run_id": "RUN_1",
        "scenario_results_for_run": [],
    }
    node = make_score_run_node(config)
    out = node(state)
    assert len(out.get("errors", [])) > 0
    assert "run_metrics" not in out or out.get("run_metrics") is None


def test_score_run_node_skips_when_errors_already():
    config = EvalAsServiceConfig()
    state: EvalAsServiceState = {
        "errors": ["Existing error"],
        "run_id": "RUN_1",
        "scenario_results_for_run": [{"scenario_id": "S1"}],
    }
    node = make_score_run_node(config)
    out = node(state)
    assert out["errors"] == ["Existing error"]
    assert "run_metrics" not in out


# -----------------------------------------------------------------------------
# Update trigger history node
# -----------------------------------------------------------------------------
def test_update_trigger_history_node_success():
    config = EvalAsServiceConfig()
    state: EvalAsServiceState = {
        "errors": [],
        "run_id": "RUN_1",
        "trigger_history": [],
        "run_metrics": {
            "run_id": "RUN_1",
            "total_scenarios": 1,
            "scenarios_passed": 1,
            "overall_pass_rate": 1.0,
            "issue_classification_accuracy": 1.0,
            "resolution_path_accuracy": 1.0,
            "outcome_accuracy": 1.0,
            "high_risk_failures": 0,
            "human_review_rate": 0.0,
            "avg_latency_ms": 100.0,
            "p95_latency_ms": 100.0,
            "max_latency_ms": 100.0,
            "hallucination_rate": 0.0,
            "policy_violation_rate": 0.0,
            "weighted_failure_rate": 0.0,
            "evaluation_score": 0.9,
            "regression_flags": [],
            "improvement_flags": [],
            "trigger_flags": [],
        },
    }
    node = make_update_trigger_history_node(config)
    out = node(state)
    assert out.get("errors") == []
    assert len(out["trigger_history"]) == 1
    assert out["current_trigger_summary"]["run_id"] == "RUN_1"
    assert out["current_trigger_summary"]["system_status"] == "healthy"


def test_update_trigger_history_node_no_metrics_appends_error():
    config = EvalAsServiceConfig()
    state: EvalAsServiceState = {
        "errors": [],
        "run_id": "RUN_1",
        "trigger_history": [],
        "run_metrics": None,
    }
    node = make_update_trigger_history_node(config)
    out = node(state)
    assert len(out.get("errors", [])) > 0


# -----------------------------------------------------------------------------
# Report node
# -----------------------------------------------------------------------------
def test_report_node_success(tmp_path):
    config = EvalAsServiceConfig()
    config.reports_dir = str(tmp_path / "reports")
    state: EvalAsServiceState = {
        "errors": [],
        "project_root": str(tmp_path),
        "run_id": "RUN_1",
        "run_metrics": {
            "run_id": "RUN_1",
            "total_scenarios": 1,
            "scenarios_passed": 1,
            "overall_pass_rate": 1.0,
            "issue_classification_accuracy": 1.0,
            "resolution_path_accuracy": 1.0,
            "outcome_accuracy": 1.0,
            "high_risk_failures": 0,
            "human_review_rate": 0.0,
            "avg_latency_ms": 100.0,
            "p95_latency_ms": 100.0,
            "max_latency_ms": 100.0,
            "hallucination_rate": 0.0,
            "policy_violation_rate": 0.0,
            "weighted_failure_rate": 0.0,
            "evaluation_score": 0.9,
            "regression_flags": [],
            "improvement_flags": [],
            "trigger_flags": [],
        },
        "current_trigger_summary": {"system_status": "healthy"},
        "scenario_scores": [
            {
                "scenario_id": "S1",
                "issue_classification_correct": True,
                "resolution_path_correct": True,
                "outcome_correct": True,
                "failure_type": "passed",
            },
        ],
    }
    node = make_report_node(config)
    out = node(state)
    assert out.get("errors") == []
    assert "report_file_path" in out
    assert out["report_file_path"]
    assert Path(out["report_file_path"]).exists()
    assert "RUN_1" in Path(out["report_file_path"]).read_text()


def test_report_node_no_metrics_appends_error():
    config = EvalAsServiceConfig()
    state: EvalAsServiceState = {
        "errors": [],
        "run_metrics": None,
    }
    node = make_report_node(config)
    out = node(state)
    assert len(out.get("errors", [])) > 0
    assert "report_file_path" not in out or out.get("report_file_path") is None
